<a href="https://colab.research.google.com/github/adasma04/Analyse_de_donnee/blob/main/analyse_clustring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


In [ ]:
from google.colab import files

uploaded = files.upload()

filename = next(iter(uploaded))

df = pd.read_csv(filename)


In [ ]:
df.head()

In [ ]:
print("Number of rows and columns:", df.shape)

In [ ]:
df.info()

In [ ]:
print(df.isnull().sum())

In [ ]:
print("Duplicated rows:", df.duplicated().sum())

In [ ]:
df.describe()

In [ ]:
# Check duplicated rows
print("Before removing duplicates:", df.shape)

# Remove duplicated rows
df = df.drop_duplicates()

print("After removing duplicates:", df.shape)

# Check duplicates again
print("Remaining duplicates:", df.duplicated().sum())

# Check missing values again
print("\nMissing values:")
print(df.isnull().sum().sum())


In [ ]:
plt.figure(figsize=(7, 5))

sns.countplot(data=df, x='Revenue')

plt.title('Distribution of Revenue')
plt.xlabel('Purchase')
plt.ylabel('Number of Visitors')

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(data=df, x='VisitorType', hue='Revenue')

plt.title('Revenue Distribution by Visitor Type')
plt.xlabel('Visitor Type')
plt.ylabel('Number of Visitors')

plt.show()

In [ ]:
monthly_purchase = df.groupby('Month')['Revenue'].mean() * 100

month_order = [
    'Feb', 'Mar', 'May', 'June',
    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'
]

monthly_purchase = monthly_purchase.reindex(month_order)

plt.figure(figsize=(10, 5))

sns.barplot(
    x=monthly_purchase.index,
    y=monthly_purchase.values
)

plt.title('Purchase Rate by Month')
plt.xlabel('Month')
plt.ylabel('Purchase Rate (%)')

plt.show()

In [ ]:
plt.figure(figsize=(12, 8))

correlation = df.select_dtypes(include='number').corr()

sns.heatmap(
    correlation,
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Correlation Heatmap')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df,
    x='Revenue',
    y='ProductRelated_Duration'
)

plt.title('Product Browsing Duration vs Purchase')
plt.xlabel('Purchase')
plt.ylabel('Product Related Duration')

plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler

features = [
    'Administrative',
    'Administrative_Duration',
    'Informational',
    'Informational_Duration',
    'ProductRelated',
    'ProductRelated_Duration',
    'BounceRates',
    'ExitRates',
    'PageValues',
    'SpecialDay'
]

X = df[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=features
)

print(X_scaled.head())
print("\nMean:")
print(X_scaled.mean().round(2))

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

wcss = []

for k in range(1, 11):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), wcss, marker='o')

plt.title('Elbow Method')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS')
plt.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

silhouette_scores = []

for k in range(2, 11):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X_scaled)

    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), silhouette_scores, marker='o')

plt.title('Silhouette Score')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.show()

In [ ]:
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

df['Cluster'] = kmeans.fit_predict(X_scaled)

print(df[['Cluster']].head())

In [ ]:
cluster_counts = df['Cluster'].value_counts().sort_index()

print(cluster_counts)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))

cluster_counts.plot(kind='bar')

plt.title('Number of Visitors in Each Cluster')
plt.xlabel('Cluster')
plt.ylabel('Number of Visitors')
plt.xticks(rotation=0)

plt.show()

In [ ]:
cluster_analysis = df.groupby('Cluster')[features].mean()

print(cluster_analysis.round(2))

In [ ]:
purchase_analysis = df.groupby('Cluster')['Revenue'].mean() * 100

print(purchase_analysis.round(2))

In [ ]:
import matplotlib.pyplot as plt

purchase_analysis.plot(kind='bar', figsize=(8, 5))

plt.title('Purchase Rate by Cluster')
plt.xlabel('Cluster')
plt.ylabel('Purchase Rate (%)')
plt.xticks(rotation=0)
plt.show()

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Reduce dimensions to 2
pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

# Create a DataFrame
pca_df = pd.DataFrame(
    X_pca,
    columns=['PC1', 'PC2']
)

pca_df['Cluster'] = df['Cluster'].values

# Plot clusters
plt.figure(figsize=(9, 6))

for cluster in sorted(pca_df['Cluster'].unique()):
    cluster_data = pca_df[pca_df['Cluster'] == cluster]

    plt.scatter(
        cluster_data['PC1'],
        cluster_data['PC2'],
        label=f'Cluster {cluster}',
        alpha=0.5
    )

plt.title('K-Means Clusters Visualization using PCA')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend()
plt.show()

In [ ]:
print("Explained Variance Ratio:")
print(pca.explained_variance_ratio_)

print("Total Explained Variance:")
print(pca.explained_variance_ratio_.sum() * 100, "%")

In [ ]:
# Statistical Analysis

statistical_analysis = df[features].agg([
    'mean',
    'median',
    'std',
    'var',
    'min',
    'max'
])

print(statistical_analysis.round(2))